# AMD-Latent
### Feature maps from Machine Learning models

---

The current notebook applies explainability approaches regarding Machine Learning models for AMD classification.

In [ ]:
# PACKAGES IMPORT

# System packages
import os
import numpy as np
from pathlib import Path
import pandas as pd
from tqdm import tqdm

# Jupyter packages
import ipywidgets as widgets
from amdlatent.utils.widgets import filechooser, toggleoptions, slideroptions

# Classification packages
import pickle as pkl
from amdlatent.models.encoders import modeldef, extrvencoder
from amdlatent.dataset.extraction import setsextraction
from amdlatent.models.classifiers import trainmodel, fitseed
from amdlatent.dataset.extraction import besthypersextraction

# Explainability packages
from amdlatent.models.explainability import heatmapscarousel, computeshap, rankdimensions
import shap

# Path checker
if Path().resolve().name != "AMD-Latent": os.chdir(os.path.join(Path().resolve().parents[1]))

# Code execution

Select the following options for running the script and execute it.

In [ ]:
# OPTIONS SELECTION

dirselect = filechooser("Select the <b>AMD Study</b> folder:")
modalityselect = toggleoptions(['Importance', 'Beeswarm', 'Heatmaps'], "Select the <b>modality</b> for script execution:")
repeatselect = slideroptions("Select the <b>number of repeats</b> to try:", 1, 10, 1, 1)

visualbox = widgets.Output()
display(visualbox)
multipleselect = heatmapselect = interestselect = None

def _update_visualbox(change=None):
    global multipleselect, heatmapselect, interestselect
    visualbox.clear_output()
    if modalityselect.value == "Heatmaps":
        with visualbox:
            heatmapselect = toggleoptions(['Value', 'Attention'], "Select which type of heatmaps <b>type of heatmaps</b> to visualize:")
            interestselect = toggleoptions(['Global', 'Specific'], "Select whether to visualize heatmaps for <b>all or specific</b> features:")
            _bind_topbox()
            multipleselect = toggleoptions(['No', 'Yes'], "Select whether to visualize <b>multiple AMD</b> cases all at once:")
    else: multipleselect, heatmapselect, interestselect = None, None, None

def _bind_visualbox(change=None):
    if modalityselect is not None:
        modalityselect.observe(_update_visualbox, names="value")
    _update_visualbox()
_bind_visualbox()

topbox = widgets.Output()
display(topbox)
featureselect = None

def _update_topbox(change=None):
    global featureselect
    topbox.clear_output()
    if interestselect.value == "Specific":
        with topbox:
            featureselect = slideroptions("Select the <b>number of top features</b> to visualize:", 0, 100, 1, 3)
    else: featureselect = None

def _bind_topbox(change=None):
    if interestselect is not None:
        interestselect.observe(_update_topbox, names="value")
    _update_topbox()

splitselect = toggleoptions(["Couples-wise", "Patient-wise", "Couples stratified-wise"],"Select the <b>dataset split type</b>:")
classifierselect = toggleoptions(['Random Forest', 'SVM Linear'], "Select the <b>model</b> to use:")

classbox = widgets.Output()
display(classbox)
classselect = None

def _update_classselect(change=None):
    global classselect
    classbox.clear_output()
    with classbox:
        classselect = toggleoptions(['Binary No', 'Binary Late'], "Select the <b>classification type</b> to use:")
    
def _bind_classselect(change=None):
    _update_classselect()
_bind_classselect()

In [ ]:
# SCRIPT EXECUTION

metadata = {
    "directory": dirselect.selected_path,
    "modality": modalityselect.value,
    "heatmaptype": heatmapselect.value if heatmapselect is not None else None,
    "interesttype": interestselect.value if interestselect is not None else None,
    "feattop": featureselect.value if featureselect is not None else None,
    "splittype": splitselect.value,
    "classtype": classselect.value,
    "classifier": classifierselect.value,
    "repeatselect": repeatselect.value,
    "multiple": multipleselect.value if multipleselect is not None else None,
}

dataset = pd.read_excel(os.path.join(metadata["directory"], "Data", "DatasetComplete.xlsx"), sheet_name='AMD')
dataset.columns = dataset.columns.astype(str).str.strip()
pathstyle = metadata["classtype"] + "_" + metadata["classtype"]
X_train, _, X_val, y_val, X_test, y_test, _, testpaths = setsextraction(dataset, metadata["classtype"], metadata["splittype"].lower().replace(' ', ''), testpath=True, encoder = "retclip")
model = pkl.load(open(os.path.join("results", "areds", "classifiers", "retclip", "models", metadata["splittype"].split('-')[0].lower().replace(' ', ''), metadata["classtype"].lower().replace(' ', ''), f"{metadata["classifier"].lower().replace(' ', '_')}.pkl"), "rb"))
hyperstyle = metadata['classtype']

last_model = model
shap_values = computeshap(model, X_test, metadata["classifier"])

if metadata["repeatselect"] > 1:
    shapperrep = [np.abs(computeshap(model, X_val, metadata["classifier"])).mean(axis=0)]
    for i in tqdm(range(int(metadata["repeatselect"]))):
        besthyper = besthypersextraction(metadata["splittype"], hyperstyle, metadata["classifier"], encoder = "retclip")
        X_train_temp, y_train_temp, X_val_temp, y_val_temp, _, _, _, _ = setsextraction(dataset, pathstyle, metadata["splittype"], mixing = True, randomstate = i, testpath = True, encoder = "retclip")
        model_temp = trainmodel(X_train_temp, y_train_temp, metadata["classifier"], besthyper, randomstate=fitseed(metadata["classtype"], metadata["classifier"], metadata["splittype"], i))
        last_model = model_temp
        shap_values_val_temp = computeshap(model_temp, X_val_temp, metadata["classifier"])
        shapperrep.append(np.abs(shap_values_val_temp).mean(axis=0))
    ranking = rankdimensions(np.vstack(shapperrep), topk=20, window=20)
    display(ranking[["Dimension", "MeanAbsSHAP", "InTop20Count"]].rename(columns={"Dimension": "Feature", "InTop20Count": "Count"}).head(20))

if metadata["modality"] == "Importance":
    shap.summary_plot(shap_values, features=X_test, plot_type="bar")
elif metadata["modality"] == "Heatmaps":
    retclipmodel, ckpt = modeldef()
    vencoder = extrvencoder(retclipmodel, ckpt)
    _, _, _, _, _, amd_test, _,  = setsextraction(dataset, "All", metadata["splittype"].lower().replace(' ', ''), encoder = "retclip")
    rows = {}
    if metadata["multiple"] == "No":
        rows[amd_test[int(np.random.randint(0, len(testpaths)))]] = int(np.random.randint(0, len(testpaths)))
    elif metadata["multiple"] == "Yes":
        rng = np.random.default_rng(42)
        for amd in rng.choice(np.unique(amd_test), size=min(6, len(np.unique(amd_test))), replace=False):
            candidates = np.flatnonzero(np.asarray(amd_test) == amd)
            rows[amd] = int(rng.choice(candidates))
    from tqdm import tqdm
    for row in tqdm(rows.values(), desc="Processing patients"):
        X = X_test[row].reshape(1, -1)
        LEpath, REpath = testpaths[row][0], testpaths[row][1]
        LEpath = os.path.join(metadata["directory"], "Data", "EDO", LEpath.split("Data/EDO/")[-1])
        REpath = os.path.join(metadata["directory"], "Data", "EDO", REpath.split("Data/EDO/")[-1])
        y_true = y_test[row]
        y_pred = model.predict(X)[0]
        shap_value = computeshap(last_model, X, metadata["classifier"])
        if metadata["interesttype"] == "Specific":
            featimportance = pd.DataFrame(
                list(zip(range(X_test.shape[1]), np.abs(shap_values).mean(axis=0))), 
                columns=['Feature_Name', 'SHAP_Importance']
            )
            featimportance = featimportance.sort_values(by='SHAP_Importance', ascending=False).reset_index(drop=True)
            bestfeats = featimportance.head(int(metadata["feattop"]))['Feature_Name'].tolist()
        else: bestfeats = None
        heatmapscarousel(LEpath, REpath, vencoder, y_pred, y_true, X, type = metadata["heatmaptype"].lower(), shap1=shap_value, feats=bestfeats)
elif metadata["modality"] == "Beeswarm":
    shap.summary_plot(shap_values, features=X_test)